<a href="https://colab.research.google.com/github/leticiabbacellar/Python/blob/main/projeto_verifica_o_de_imagem_gerada_por_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pillow --quiet

import os
import hashlib
import sqlite3
import requests
from PIL import Image
from google.colab import userdata, files


try:
    API_USER = userdata.get('SIGHTENGINE_API_USER')
    API_SECRET = userdata.get('SIGHTENGINE_API_SECRET')
except Exception:
    print("ERRO: Configure as chaves SIGHTENGINE_API_USER e SIGHTENGINE_API_SECRET no menu de chaves (🔑) do Colab.")

# Inicializa o banco de dados local
DB_NAME = "eleicoes_cache.db"
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS cache_imagens
    (hash TEXT PRIMARY KEY, veredito TEXT, probabilidade_ia REAL)
''')
conn.commit()


In [ ]:
# FUNÇÕES CORE DO PROJETO

def calcular_hash_imagem(caminho_imagem):
    """Gera uma assinatura digital única baseada nos bytes da imagem"""
    hasher = hashlib.sha256()
    with open(caminho_imagem, 'rb') as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

def consultar_sightengine(caminho_imagem):
    """Faz a chamada real para a API do Sightengine utilizando o modelo de IA"""
    url = 'https://api.sightengine.com/1.0/check.json'

    params = {
        'models': 'genai',
        'api_user': API_USER,
        'api_secret': API_SECRET
    }

    with open(caminho_imagem, 'rb') as img_file:
        files_payload = {'media': img_file}
        response = requests.post(url, files=files_payload, data=params)

    if response.status_code == 200:
        dados = response.json()

        # Ajuste exato para ler a estrutura {'type': {'ai_generated': 0.99}}
        if 'type' in dados and 'ai_generated' in dados['type']:
            score = dados['type']['ai_generated']
            probabilidade = score * 100
            veredito = "IA" if probabilidade >= 50 else "REAL"
            return veredito, round(probabilidade, 2)
        else:
            print("Resposta inesperada da API:", dados)
            return "ERRO_ANALISE", 0.0
    else:
        print(f"Erro na requisição HTTP: {response.status_code}")
        print(response.text)
        return "ERRO_CONEXAO", 0.0

def analisar_imagem_eleitoral(caminho_imagem):
    """Gerencia o fluxo de verificação (Cache local -> API)"""
    if not os.path.exists(caminho_imagem):
        print(f"Arquivo {caminho_imagem} não encontrado.")
        return

    # 1. Calcula a assinatura digital da imagem
    hash_foto = calcular_hash_imagem(caminho_imagem)

    # 2. Verifica se a imagem já foi analisada antes (Otimização de limite/crédito)
    cursor.execute("SELECT veredito, probabilidade_ia FROM cache_imagens WHERE hash=?", (hash_foto,))
    resultado_cache = cursor.fetchone()

    if resultado_cache:
        print("\n⚡ [CACHE LOCAL] Esta imagem já foi analisada anteriormente!")
        print(f"Veredito: {resultado_cache[0]} | Confiança de IA: {resultado_cache[1]}%")
        return

    # 3. Se for inédita, envia para a API na internet
    print("\n🌐 [API EXTERNA] Imagem nova detectada. Consultando Sightengine...")
    veredito, probabilidade = consultar_sightengine(caminho_imagem)

    if "ERRO" not in veredito:
        # 4. Salva no banco de dados para não gastar créditos se a imagem reaparecer
        cursor.execute("INSERT INTO cache_imagens VALUES (?, ?, ?)", (hash_foto, veredito, probabilidade))
        conn.commit()
        print(f"Análise Concluída! Veredito: {veredito} | Confiança de IA: {probabilidade}%")
        print("Resultado salvo no banco local para futuras consultas.")
    else:
        print("Não foi possível classificar a imagem devido a um erro.")

# INTERFACE DE EXECUÇÃO

#print("Faça o upload de uma imagem para testar o sistema:")
#upload = files.upload()

#for nome_arquivo in upload.keys():
 #   print(f"\nIniciando análise do arquivo: {nome_arquivo}")
  #  analisar_imagem_eleitoral(nome_arquivo)

In [ ]:
!pip install gradio

In [ ]:
def rodar_para_interface(caminho_imagem):
    if caminho_imagem is None:
        return "⚠️ **Nenhuma imagem foi enviada.** Por favor, selecione um arquivo."

    if not os.path.exists(caminho_imagem):
        return f"⚠️ **Arquivo `{caminho_imagem}` não encontrado no servidor.**"

    try:

        conn_local = sqlite3.connect("cache_imagens.db")
        cursor_local = conn_local.cursor()

        cursor_local.execute('''
            CREATE TABLE IF NOT EXISTS cache_imagens
            (hash TEXT PRIMARY KEY, veredito TEXT, probabilidade_ia REAL)
        ''')
        conn_local.commit()

        hash_foto = calcular_hash_imagem(caminho_imagem)

        cursor_local.execute(
            "SELECT veredito, probabilidade_ia FROM cache_imagens WHERE hash=?",
            (hash_foto,)
        )
        resultado_cache = cursor_local.fetchone()

        if resultado_cache:
            veredito, probabilidade = resultado_cache
            origem = "⚡"
        else:
            veredito, probabilidade = consultar_sightengine(caminho_imagem)

            if "ERRO" in veredito:
                conn_local.close()
                return f"❌ **Falha no processamento  ({veredito}).** Verifique suas credenciais."

            cursor_local.execute(
                "INSERT INTO cache_imagens VALUES (?, ?, ?)",
                (hash_foto, veredito, probabilidade)
            )
            conn_local.commit()
            origem = "🌐 "

        conn_local.close()


        alerta = "🚨 **ALERTA DE CONTEÚDO SINTÉTICO (IA)**" if veredito == "IA" else "✅ **MÍDIA APARENTEMENTE REAL**"

        texto_resultado = f"""
### {alerta}

* **Veredito:** `{veredito}`
* **Probabilidade de IA:** `{probabilidade}%`


---
{origem}
        """
        return texto_resultado

    except Exception as e:
        return f"💥 **Erro interno ao processar a imagem:** `{str(e)}`"

In [ ]:
import gradio as gr
import requests

with gr.Blocks(title="Sistema VERIX") as demo:
    gr.Markdown("# 🗳️ Sistema VERIX - Sistema de Verificação de Imagens Eleitorais")

    with gr.Row():
        with gr.Column():
            entrada_imagem = gr.Image(type="filepath", label="Envie a imagem")
            botao = gr.Button("Analisar Imagem", variant="primary")

        with gr.Column():
            saida_texto = gr.Markdown(label="Resultado")


    botao.click(
        fn=rodar_para_interface,
        inputs=entrada_imagem,
        outputs=saida_texto
    )

# Abre o link externo público
demo.launch(share=True, debug=True)